# **Data Cleaning**

**MSc Thesis Euan Bronsky**

**Notebook Number: 1**

This notebook focuses on cleaning the data, where particular attention is drawn to arbitrage consistency of the final dataset. Standard filters from the literature are applied after which a no-arbitrage algorithm is applied. Moreover, the option implied volatilities and deltas are computed after which the quotes are classified into several moneyness-maturity buckets.

----------------------------

- **Import dependencies**

In [1]:
%reset -f
import pandas as pd
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import optimize
from mpl_toolkits.mplot3d import Axes3D
import requests
import certifi
from io import StringIO

- **Read the data**

In [2]:
# Collect the files globally
files = glob.glob(f"/Users/euanbronsky/Downloads/Options.SPX.20*/*.txt")

# Read the files and concatenate into a dataframe
data = pd.concat([pd.read_csv(file, sep = ',', engine = 'c', low_memory=False) for file in files], ignore_index=True)

- **Preliminary adjustments**

In [3]:
# Clean column names
data.columns = data.columns.str.lower().str.replace(r'[\[\]\s]','', regex=True)

# Adjust columns and keep only relevant columns
relevant_cols = ['quote_date', 'underlying_last', 'expire_date', 'dte', 'c_iv', 'c_bid', 'c_ask', 'c_delta', 'strike', 'p_bid', 'p_ask', 'p_iv', 'p_delta']
data = data[relevant_cols].copy()

# Convert datatypes in one go using assign function
data = data.assign(
    **{col: pd.to_numeric(data[col], errors='coerce')
       for col in ['underlying_last', 'dte', 'c_iv', 'c_bid', 'c_ask', 'c_delta', 'strike', 'p_bid', 'p_ask', 'p_iv', 'p_delta']},
    **{col: pd.to_datetime(data[col], errors='coerce')
       for col in ['quote_date', 'expire_date']})

- **Separate calls from puts and merge them into a long format dataframe**

In [4]:
# Define common columns
common_cols = ['quote_date', 'underlying_last', 'expire_date', 'dte', 'strike']

# Handle calls separately
calls = (data[common_cols + ['c_iv', 'c_bid', 'c_ask', 'c_delta']]
         .rename(columns={'c_bid': 'bid', 'c_ask': 'ask', 'c_iv': 'iv', 'c_delta': 'delta'})
         .assign(option_type = 'call'))

# Separate puts from calls
puts = (data[common_cols + ['p_iv', 'p_bid', 'p_ask', 'p_delta']]
        .rename(columns={'p_bid': 'bid', 'p_ask': 'ask', 'p_iv': 'iv', 'p_delta': 'delta'})
        .assign(option_type = 'put'))

# Concatenate the results and sort values
df = pd.concat([calls, puts], ignore_index=True)
df = df.sort_values(by = ['option_type', 'quote_date', 'expire_date', 'strike'])

# **1. Cleaning function**

---------------------------------

The cleaning of data is crucial to ensure models are not trained on low-liquidity quotes, as these quotes may contain stale prices, lage bid-ask spreads, or pricing errors. Including such observations can distort the estimated implied volatility surface and lead to unreliable factor estimates. By filtering the data, the analysis focuses on more reliable and actively traded option quotes. This improves the quality of estimated models and makes the resulting volatility dynamics more representative of actual market dynamics.



- **Cleaning function**

Computation of midquotes $\text{midquote} = \frac{bid + ask}{2}$, removal of low prices $\text{midquote} \leq 3/8$, filter for moderate time-to-maturities $7 \leq \text{TTM} \leq 360 $, and remove ITM options: $|\Delta| > 0.5$. The function includes the number of observations left after removal, the drop count, and percentage drop.

In [5]:
# initial cleaning function
def cleaning_fun(df):

    # Copy the dataframe and initiate dictionary
    df = df.copy()
    stats = {}

    # Initial count
    stats['initial'] = len(df)

    # Step 1: compute mid prices
    df['mid'] = (df['bid'] + df['ask'])/2
    stats['after_mid'] = len(df)

    # Step 2: remove low-priced (DOTM) options
    df = df[df['mid'] >= 3/8]
    stats['after_price'] = len(df)

    # Step 3: removal of very short and long TTM
    df= df[(df['dte'] >= 7) & (df['dte'] <= 360)]
    stats['after_ttm'] = len(df)

    # Step 4: removal of ITM options
    df = df[
    ((df['option_type'] == 'call') & (df['delta'] <= 0.5)) |
    ((df['option_type'] == 'put') & (df['delta'] >= -0.5))]
    stats['after_moneyness'] = len(df)

    # Step 5: remove IV quotes above 1
    df = df[df['iv'] <= 1]
    stats['after_iv'] = len(df)

    # Convert to dropped counts
    drops = {
        'price_drop': stats['initial'] - stats['after_price'],
        'ttm_drop': stats['after_price'] - stats['after_ttm'],
        'moneyness_drop': stats['after_ttm'] - stats['after_moneyness'],
        'iv_drop': stats['after_moneyness'] - stats['after_iv']}

    # Convert to percentage dropped
    percent_drops = {
        'price_drop_pct': (drops['price_drop'] / stats['initial']) * 100,
        'ttm_drop_pct': (drops['ttm_drop'] / stats['after_price']) * 100,
        'moneyness_drop_pct': (drops['moneyness_drop'] / stats['after_ttm']) * 100,
        'iv_drop_pct': (drops['iv_drop'] / stats['after_moneyness']) * 100}

    # Return the dataframe
    return df, stats, drops, percent_drops

- **Apply the cleaning function**

In [6]:
# Apply the function
df1, stats, drops, percent_drops = cleaning_fun(df)

# Create dataframe with statistics and print
df1_stats = pd.DataFrame({
    'step': ['price', 'ttm', 'moneyness', 'iv'],
    'remaining': [
        stats['after_price'],
        stats['after_ttm'],
        stats['after_moneyness'],
        stats['after_iv']],
    'dropped': [
        drops['price_drop'],
        drops['ttm_drop'],
        drops['moneyness_drop'],
        drops['iv_drop']],
    'pct_drop': [
        percent_drops['price_drop_pct'],
        percent_drops['ttm_drop_pct'],
        percent_drops['moneyness_drop_pct'],
        percent_drops['iv_drop_pct']]})
df1_stats

,step,remaining,dropped,pct_drop
0,price,27690773,3422181,10.999216
1,ttm,23331733,4359040,15.741850
2,moneyness,10508485,12823248,54.960547
3,iv,10502621,5864,0.055803


# **2. Dividend yields**

--------------------------------------

Dividend yields are a key input for the Black-Scholes-Merton equation, as they affect the theoretical value of options written on dividend-paying assets. Ignoring dividends can lead to biased option prices, especially for longer-maturity options or assets with substantial dividend payments.



- **Read the data**

In [ ]:
# Import monthly dividend data
url = "https://www.multpl.com/s-p-500-dividend-yield/table/by-month"
html = requests.get(url, verify=certifi.where()).text
div_df = pd.read_html(StringIO(html))[0]
df1 = df1.reset_index(drop=True)

- **Import the dividend data, filter, and merge into the existing dataframe**

In [7]:
# Rename the columns
div_df.columns = ["date", "div_value"]

# Set to datetime and adjust strings
div_df["date"] = pd.to_datetime(div_df["date"])
div_df["div_value"] = (
    div_df["div_value"]
    .str.replace("%", "", regex=False)
    .str.replace("†", "", regex=False)
    .str.strip()
    .astype(float))

# Filter the data
div_df = div_df[(div_df['date'] >= '2010-01-04') & (div_df['date'] <= '2024-01-29')]
div_df = div_df[::-1].reset_index(drop=True)

# Copy the dataframe
df_extra = df1.copy()

# Extract year and months to match
df_extra['year_month'] = df_extra['quote_date'].dt.to_period('M')
div_df['year_month'] = div_df['date'].dt.to_period('M')

# Merge the dataframes
df_extra = df_extra.merge(div_df[['div_value', 'year_month']], left_on = 'year_month', right_on = 'year_month', how = 'left')

# Compute the monthly annualized yields
df1['q'] = df_extra['div_value'] / 100

## **3. Risk-free rate**

---------------------------

The risk-free rate is another key input for the Black-Scholes-Merton equation, as it affects the discounting of the option payoff and the forward value of the underlying asset. Including an appropriate risk-free rate ensures that option prices are values consistently over different maturities, improving the reliability of the estimated implied volatiluty surface.

- **Read the data**

In [8]:
# Import the treasury data
rf_data = pd.read_csv('/Users/euanbronsky/Downloads/yield-curve-rates-1990-2023.csv')

- **Adjust the data**

In [9]:
# Convert to datetime, reverse index, and reset index
rf_data = (rf_data
           .assign(Date = pd.to_datetime(rf_data['Date'], errors='coerce', format='%m/%d/%y'))
           .iloc[::-1]
           .reset_index(drop=True))

# Adjust the columns
rf_data.columns = rf_data.columns.str.lower().str.replace(r'\s', '', regex=True)

# Operationalize the rates and compute T in the original DF
cols = rf_data.columns.drop('date')
rf_data[cols] = rf_data[cols] / 100
df1['T'] = df1['dte'] / 365

- **Merge the data and classify**

In [10]:
# Merge the dataframes
df1 = df1.reset_index(drop=True)
aux_df = df1.merge(rf_data[['date'] + cols.tolist()], left_on='quote_date', right_on = 'date', how='left')

# Approximate the closest maturity
df1['r'] = np.select([
    df1['dte'] <= 45,
    df1['dte'] <= 75,
    df1['dte'] <= 105,
    df1['dte'] <= 150,
    df1['dte'] <= 270],
[
    aux_df['1mo'],
    aux_df['2mo'].fillna(aux_df['1mo']).fillna(aux_df['3mo']),
    aux_df['3mo'],
    aux_df['4mo'].fillna(aux_df['3mo']).fillna(aux_df['6mo']),
    aux_df['6mo']], default = aux_df['1yr'])

# **4. No-arbitrage filtering**

----------------------------------------------

Ensuring no-arbitrage in the initial set of data is crucial, as arbitrage violations may indicate mispriced, stale, or unreliable option quotes. If such observations are included, they can distort the estimated implied volatility surface and lead to misleading model results. By removing quotes that violate basic no-arbitrage conditions, the analysis is based on prices that are economically consistent.


- **Filter function**

To ensure arbitrage consistency, several conditions are enforced, under which the upper and lower bounds: $C_t \geq (S_te^{-q(T-t)} - Ke^{-r(T-t)})^+ \quad$, $C_t \leq S_te^{-q(T-t)} \quad$, monotonicity in strike: for $K_1 < K_2 < \cdot \cdot \cdot < K_n \quad$, $C_t(K_1, T) \geq C_t(K_2, T) \geq \cdot \cdot \cdot \geq C_t(K_n, T)$, convexity in strike: $\frac{C_t(K_i, T) - C_t(K_{i-1}, T)}{K_i - K_{i-1}} \leq \frac{C_t(K_{i+1}, T) - C_t(K_i, T)}{K_{i+1} - K_i}$, and alendar monotonicity: $P_t(K, T_1) \leq P_t(K, T_2) \quad $ whenever $T_1 < T_2$

In [11]:
# Arbitrage filter
def arbitrage_filter(df1, run_bounds=True):

    # Copy dataframe
    df1 = df1.copy()

    # Initiate dictionary
    stats = {}
    stats['initial'] = len(df1)

    # If statement
    if run_bounds:

        # Step 1a: Put and call lower bounds
        df1 = df1[((df1['option_type'] == 'call') & (df1['mid'] >= np.maximum(df1['underlying_last']*np.exp(-df1['q']*df1['T']) - df1['strike']*np.exp(-df1['r']*df1['T']),0)) | (df1['option_type'] == 'put') & (df1['mid'] >= np.maximum(df1['strike']*np.exp(-df1['r']*df1['T']) - df1['underlying_last']*np.exp(-df1['q'] * df1['T']), 0)))]

        # Step 1b: Put and call upper bounds
        df1 = df1[((df1['option_type'] == 'call') & (df1['mid'] <= df1['underlying_last'] * np.exp(-df1['q'] * df1['T']))) | (df1['option_type'] == 'put') & (df1['mid'] <= df1['strike']*np.exp(-df1['r']*df1['T']))]
    stats['after_bounds'] = len(df1)

    # Sort by strike
    df1 = df1.sort_values(['option_type', 'quote_date', 'expire_date', 'strike'])

    # Step 2: Monotonicity in strike
    diffs = df1.groupby(['option_type', 'quote_date', 'expire_date'])['mid'].diff()
    df1 = df1[((df1['option_type'] == 'call') & (diffs.isna() | (diffs <= 0))) | ((df1['option_type'] == 'put') & (diffs.isna() | (diffs >= 0)))]
    stats['after_strike_mono'] = len(df1)

    # Step 3: Strike convexity
    slopes = (df1.groupby(['option_type', 'quote_date', 'expire_date'])['mid'].diff() / df1.groupby(['option_type', 'quote_date', 'expire_date'])['strike'].diff())
    slope_diffs = slopes.groupby([df1['option_type'], df1['quote_date'], df1['expire_date']]).diff()
    viol = (slope_diffs < 0).groupby([df1['option_type'], df1['quote_date'], df1['expire_date']]).shift(-1)
    viol = viol.fillna(False).astype(bool)
    df1 = df1.loc[~viol]
    stats['after_strike_convex'] = len(df1)

    # Step 4: Calendar monotonicity
    df1 = df1.sort_values(['option_type', 'quote_date', 'strike', 'dte'])
    diffs_ttm = df1.groupby(['option_type', 'quote_date', 'strike'])['mid'].diff()
    df1 = df1[
        (df1['option_type'] == 'call') |
        ((df1['option_type'] == 'put') & (diffs_ttm.isna()| (diffs_ttm >= 0)))]
    stats['after_calendar_mono'] = len(df1)

    # Re-sort the dataframe and reset the index
    df1 = (df1
           .sort_values(['option_type', 'quote_date', 'expire_date', 'strike'])
           .reset_index(drop=True))

    # Compute number drops per steps
    drops = {
        'ub_lb_drop': stats['initial'] - stats['after_bounds'],
        'strike_mono_drop': stats['after_bounds'] - stats['after_strike_mono'],
        'convexity_drop': stats['after_strike_mono'] - stats['after_strike_convex'],
        'calendar_mono_drop': stats['after_strike_convex'] - stats['after_calendar_mono'],
    }

    # Compute percentage drops per step
    pct_drops = {
        'bounds_pct': drops['ub_lb_drop'] / stats['initial'] * 100,
        'strike_mono_pct': drops['strike_mono_drop'] / stats['after_bounds'] * 100,
        'strike_convex_pct': drops['convexity_drop'] / stats['after_strike_mono'] * 100,
        'calendar_mono_pct': drops['calendar_mono_drop'] / stats['after_strike_convex'] * 100,
    }

    # Return the results
    return df1, stats, drops, pct_drops

- **Apply the function and display the results**

In [12]:
# Apply the function and retrieve the results
df_help, stats, drops, pct_drops = arbitrage_filter(df1, run_bounds=True)

# Construct dataframe and display
df_arbitrage = pd.DataFrame({
    'Step': ['Bounds', 'Strike_mono', 'Strike_convex', 'Calendar_mono'],
    'Obs. remaining': [
        stats['after_bounds'],
        stats['after_strike_mono'],
        stats['after_strike_convex'],
        stats['after_calendar_mono']],
    'Obs. removed': [
        drops['ub_lb_drop'],
        drops['strike_mono_drop'],
        drops['convexity_drop'],
        drops['calendar_mono_drop']],
    'Pct. removed': [
        pct_drops['bounds_pct'],
        pct_drops['strike_mono_pct'],
        pct_drops['strike_convex_pct'],
        pct_drops['calendar_mono_pct']],
})
df_arbitrage

,Step,Obs. remaining,Obs. removed,Pct. removed
0,Bounds,10367444,135177,1.287079
1,Strike_mono,10161861,205583,1.982967
2,Strike_convex,7036346,3125515,30.757309
3,Calendar_mono,7024132,12214,0.173584


- **Loop until convergence**

Enforcing the no-arbitrage conditions requires iterating over the dataset until no further violations are detected:

$\mathcal Q^{(K)} \approx \mathcal Q^{AF} \quad$, where $ \mathcal{Q}^{AF} = \arg \max_{\mathcal{Q}' \subseteq \mathcal{Q}} |\mathcal{Q}'| $

In [13]:
# Initial specifications
prev_len = -1
max_iter = 100
i = 0

# Loop until convergence
while len(df_help) != prev_len and i < max_iter:

    # Set length to dataframe length
    prev_len = len(df_help)

    # Apply the filter without the upper and lower bound steps
    df_help, _, _, _ = arbitrage_filter(df_help, run_bounds=False)

    # Print the iteration number, previous length and current length, and number of observations dropped
    i += 1
    print(i, prev_len, len(df_help), prev_len - len(df_help))

1 7024132 6016028 1008104
2 6016028 5774278 241750
3 5774278 5726834 47444
4 5726834 5717789 9045
5 5717789 5715479 2310
6 5715479 5714577 902
7 5714577 5714122 455
8 5714122 5713831 291
9 5713831 5713631 200
10 5713631 5713481 150
11 5713481 5713368 113
12 5713368 5713283 85
13 5713283 5713222 61
14 5713222 5713180 42
15 5713180 5713151 29
16 5713151 5713130 21
17 5713130 5713113 17
18 5713113 5713099 14
19 5713099 5713087 12
20 5713087 5713080 7
21 5713080 5713077 3
22 5713077 5713075 2
23 5713075 5713073 2
24 5713073 5713071 2
25 5713071 5713069 2
26 5713069 5713067 2
27 5713067 5713065 2
28 5713065 5713063 2
29 5713063 5713061 2
30 5713061 5713059 2
31 5713059 5713057 2
32 5713057 5713055 2
33 5713055 5713054 1
34 5713054 5713053 1
35 5713053 5713053 0


# **5. Compute Implied Volatility**

--------------------------------------------

**Implied Volatility** is defined as the value of $\sigma$, holding all other inputs fixed, that equates the **Black-Scholes** option price to the observed market price: $C_t^{BS}(S_t, K, r, \tau, \sigma_{IV}) = C_t^{mkt}(S_t, K, \tau)$. This requires the Black-Scholes determined call option and put option values: $C_t = S_te^{-q(T-t)} \Phi(d_1) - K e^{-r(T-t)}\Phi(d_2)$, $P_t = Ke^{-r(T-t)} \Phi(-d_2) - S_t e^{-q(T-t)}\Phi(-d_1)$. IV is useful since it expresses option prices in a standardized volatility measure. As a result, options with different strikes and maturities can be compared more directly, allowing the IVS to be constructed and analyzed.

- **Implied volatility computation function**

The implied volatilities are obtained by numerical root solving: $    f(\sigma) =  C_t^{BS}(S_t, K, r, \tau, \sigma) - C_t^{mkt}(S_t, K, \tau)$

In [14]:
# Implied volatility function
def implied_vol(T, S, K, r, q, mkt_price, call = True):

    # Black-Scholes option price function
    def BS_price(sigma):

        # d1 and d2
        d1 = (np.log(S / K) + (r - q + 0.5* sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)

        # Compute call and put prices
        call_price = S * np.exp(-q*T) * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)
        put_price = K * np.exp(-r*T) * norm.cdf(-d2) - S * np.exp(-q * T)* norm.cdf(-d1)

        # Determine whether input is a call or put
        bs_price = call_price if call else put_price

        # Compute distance to the observed market price
        fx = bs_price - mkt_price

        # Return the difference
        return fx

    # Minimize the distance using brentq
    return optimize.brentq(BS_price, 0.001, 10, maxiter=1000)

- **Apply the function**

In [15]:
# Apply the function and store the results
df_help['iv_fun'] = df_help.apply(
    lambda row: implied_vol(
        S=row['underlying_last'],
        K = row['strike'],
        T=row['T'],
        r=row['r'],
        q=row['q'],
        mkt_price=row['mid'],
        call=(row['option_type'] == 'call')), axis=1)

# **6. Compute Deltas**

-------------------------------------------------

**Delta ($\Delta$)** is the sensitivity of the option price to a small change in the underlying asset price, $\Delta = \frac{\partial V}{\partial S}$. A positive delta means the option price increases when the underlying price rises, while a negative delta means the option price decreases. In particular, $\Delta^C_t = e^{-q(T-t)}\Phi(d_1)$ and $\Delta^P_t = \Delta^C_t - e^{-q(T-t)}$, where

$$d_1 = \frac{\log(S_t / K) + (r - q + 0.5 \sigma^2) (T-t)}{\sigma \sqrt{T-t}}$$


- **Computation of delta function**

In [16]:
# Delta function
def compute_delta(S, K, r, q, sigma, T, call = True):

    # Compute d1
    d1 = (np.log(S / K) + (r - q + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))

    # Compute the call and put delta values
    call_delta = np.exp(-q*T) * norm.cdf(d1)
    put_delta = call_delta - np.exp(-q*T)

    # Check if a call or put
    delta = call_delta if call else put_delta

    # Return the results
    return delta

- **Apply the function**

In [17]:
# Create new column with computed deltas
df_help['delta_fun'] = df_help.apply(
lambda row: compute_delta(
    row['underlying_last'],
    row['strike'],
    row['r'],
    row['q'],
    row['iv_fun'],
    row['T'],
    row['option_type'] == 'call'), axis=1)

- **Miscellaneous clean-up**

Removal of previously computed $\Delta > 0.5$, and implied volatilities $\sigma_{i,t} > 1$

In [ ]:
# Number of observations
n_start = len(df_help)

# Clean up the deltas
df_clean = df_help[((df_help['option_type'] == 'call') & (df_help['delta_fun'] <= 0.5)) | ((df_help['option_type'] == 'put') & (df_help['delta_fun'] >= -0.5))]

# Clean up the implied volatilities
df_clean = df_clean[df_clean['iv_fun'] <= 1]
n_after_iv = len(df_clean)

# Summary table
df_arbitrage2 = pd.DataFrame({
    'Step': ['Clean-up'],
    'Obs. remaining': [n_after_iv],
    'Obs. removed': [n_start - n_after_iv],
    'Pct. removed': [
        100 * (n_start - n_after_iv) / n_start]
})
df_arbitrage2

- **Save the data as a parquet file**

In [33]:
# Save as parquet file to avoid re-cleaning
df_clean.to_parquet('final_cleaned.parquet')
df_clean.to_parquet("/Users/euanbronsky/Downloads/final_cleaned.parquet")

# **7. Create buckets**

-------------------------------------------------

Since the final dataset contains over 5 million observations, it is very inefficient to estimate the models based on the entire sample. This motivates a bucketed approach, where the data are divided into 24 different moneyness-maturity clusters. Subsequently, one representative contract is selected for each cluster for each day. This reduces the computational burden while still preserving the main structure of the IVS.

"- **Moneyness buckets**

**Moneyness groups:** ATM call: $0.375 < \Delta^c_t < 0.5 \quad$, OTM call: $0.125 < \Delta^c_t < 0.375 \quad$, DOTM call: $0 < \Delta^c_t < 0.125 \quad$, DOTM put: $-0.125 < \Delta^p_t < 0 \quad$, OTM put:  $-0.375 < \Delta^p_t < -0.125 \quad$, ATM put: $-0.5 < \Delta^p_t < -0.375$.

In [34]:
# Copy dataframe
df_buckets = pd.read_parquet("/Users/euanbronsky/Downloads/final_cleaned.parquet")

# Extract delta
delta = df_buckets['delta_fun']

# Assign labels
df_buckets['moneyness_group'] = np.select(
    [(delta > 0.375) & (delta < 0.5),
    (delta > 0.125) & (delta <= 0.375),
    (delta > 0) & (delta <= 0.125),
    (delta > -0.125) & (delta <= 0),
    (delta > -0.375) & (delta <= -0.125),
    (delta > -0.5) & (delta <= -0.375)],
    ['ATM_call', 'OTM_call', 'DOTM_call', 'DOTM_put', 'OTM_put', 'ATM_put'],
    default = 'outside_range')

- **Maturity buckets**

**Maturity groups:** short: $7-45 \quad$, medium short: $45-90 \quad$, medium long: $90-180 \quad$, long: $180-360$

In [35]:
# Extract time to maturity
dte = df_buckets['dte']

# Assign labels
df_buckets['maturity_group'] = np.select(
    [dte <= 45,
     dte <= 90,
     dte <= 180,
     dte > 180],
    ['7-45', '45-90', '90-180', '180-360'], default = 'NA')

- **Midpoints and distance**

The distance is defined as the metric scaled by its variance to ensure comparability: $$
\frac{(\Delta_{i,t} - \bar \Delta_g)^2}{\sigma_\Delta^2}
+
\frac{(\tau_{i,t} - \bar \tau_g)^2}{\sigma_\tau^2}
.$$

In [36]:
# Extract relevant variables
moneyness_group = df_buckets['moneyness_group']
maturity_group = df_buckets['maturity_group']

# Assign midpoints to the moneyness groups
df_buckets['delta_star'] = np.select(
    [moneyness_group == 'ATM_call',
     moneyness_group == 'OTM_call',
     moneyness_group == 'DOTM_call',
     moneyness_group == 'DOTM_put',
     moneyness_group == 'OTM_put',
     moneyness_group == 'ATM_put'],
    [0.4375, 0.25, 0.0625, -0.0625, -0.25, -0.4375], default = np.nan
)

# Assign midpoints to the maturity groups
df_buckets['t_star'] = np.select(
    [maturity_group == '7-45',
     maturity_group == '45-90',
     maturity_group == '90-180',
     maturity_group == '180-360'],
    [26, 67.5, 135, 270], default = np.nan
)

# Compute standard deviations
sigma_delta = df_buckets['delta_fun'].std()
sigma_t = df_buckets['dte'].std()

# Compute distance
df_buckets['distance'] = (df_buckets['delta_fun'] - df_buckets['delta_star'])**2 / sigma_delta**2 + (df_buckets['dte'] - df_buckets['t_star'])**2 / sigma_t**2

- **Save the data and make extra adjustments**

In [5]:
# Adjust the DataFrame
df_buckets = df_buckets.assign(
    forward_price = np.exp(df_buckets['r'] * df_buckets['T']) * df_buckets['underlying_last'],
    M=lambda x: np.log(x['strike'] / x['forward_price']) / np.sqrt(x['T']),
    log_iv=np.log(df_buckets['iv_fun']),
    moneyness_squared=lambda x: x['M'] **2,
    interaction= lambda x: x['T'] * x['M'],
    constant=1
).sort_values(['quote_date', 'moneyness_group', 'maturity_group']).reset_index(drop=True)

# Save as parquet file to avoid re-cleaning
df_buckets.to_parquet('data_final_final.parquet')
df_buckets.to_parquet("/Users/euanbronsky/Downloads/data_final_final.parquet")